In [ ]:
from Libraries.inference_training import Configuration, ImageDataset
from Libraries.inference_training import initCudaEnvironment, createTransforms
from Libraries.inference_training import drawImageAndFeatureMasks
from Libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from Libraries.inference_training import trainModel, saveModel, loadModel
from Libraries.inference_training import createModelInstance, testInference
from Libraries.engine import evaluate
import Libraries.utils as utils
import torch
import os
import random
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[1]))
from paths import EVAL, MODEL_PATH, TEST, TRAIN

In [ ]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

# load model

In [ ]:
def load_model(path:str, config=None):
    """
    Load a trained PyTorch instance segmentation model from disk.

    This function initializes a model using the provided or default configuration and loads
    the saved model weights from the specified path.

    Parameters:
    -----------
    path : str
        Path to the saved PyTorch model.
    config : Configuration, optional
        Predefined Configuration object. If not provided, a default configuration will be created.

    Returns:
    --------
    model : torch.nn.Module
        The loaded PyTorch model ready for inference or further training.
    config : Configuration
        The Configuration object used to initialize the model.

    Notes:
    ------
    - If `config` is None, a new Configuration object will be initialized.
    - The function assumes the model architecture is defined in accordance with the Configuration.
    - Ensure the configuration matches the model's training parameters to avoid shape mismatches.
    """
    if not config:
        config = Configuration()
    
    model = createModelInstance(config)
    loadModel(config, model, path)
    
    return model, config

In [ ]:
def load_default_config():
    """
    Create and return a default Configuration object with preset parameters.

    This function initializes a Configuration instance with standard settings suitable for
    instance segmentation training and inference, including input sizes, model info, ONNX metadata,
    and legend entries.

    Parameters:
    -----------
    None

    Returns:
    --------
    config : Configuration
        A Configuration object initialized with default values for model training and export.

    Notes:
    ------
    - The default input image size is set to 250x250 pixels.
    - The model is configured for 3 input channels and 3 classes (including background).
    - ONNX metadata thresholds are set to typical defaults for score, mask, and stride.
    - The legend includes a "Background" entry with a transparent color.
    """
    config = Configuration()
    config.setIsCrowd(False)
    config.setFilePrefix("")
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2+1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.addLegendEntry("Background", 0, "#00000000")
    config.setOnnxMetaData(scoreThreshold=0.2,
                           maskThreshold=0.3,
                           strideFraction=0.5)
    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    
    return config

# average recall/average precision evaluation based on different IOU prerequisites template

In [ ]:
model_path = MODEL_PATH
eval_path = EVAL
config = load_default_config()
config.setDatasetPaths(testPath=eval_path)
evalDataset = ImageDataset(config, False, createTransforms(False))
config = load_default_config()
model, config = load_model(model_path,config)
testDataLoader = torch.utils.data.DataLoader(
    evalDataset,
    batch_size=1,
    shuffle=True,
    collate_fn=utils.collate_fn
)
evaluate(model, testDataLoader, device=config.device)

TODO: DICE score evaluation template, mAP score evaluation template

# 25 epoch combo model evaluation

In [ ]:
model_path = MODEL_PATH
trainPath = TRAIN
eval_path = EVAL
config = load_default_config()
config.setDatasetPaths(trainPath= trainPath, testPath=eval_path)
evalDataset = ImageDataset(config, False, createTransforms(False))
config = load_default_config()
model, config = load_model(model_path,config)
testDataLoader = torch.utils.data.DataLoader(
    evalDataset,
    batch_size=1,
    shuffle=True,
    collate_fn=utils.collate_fn
)
evaluate(model, testDataLoader, device=config.device)

# 25 epoch augmented combo model evaluation

In [ ]:
model_path = MODEL_PATH
trainPath = TRAIN
eval_path = EVAL
config = load_default_config()
config.setDatasetPaths(trainPath=trainPath, testPath=eval_path)
evalDataset = ImageDataset(config, False, createTransforms(False))
config = load_default_config()
model, config = load_model(model_path,config)
testDataLoader = torch.utils.data.DataLoader(
    evalDataset,
    batch_size=1,
    shuffle=True,
    collate_fn=utils.collate_fn
)
evaluate(model, testDataLoader, device=config.device)